# Notebook 10 — Native Long-Context Video vs DIY Pipeline

Closing the lab. We have a perfectly working video search engine from nb09 — frame index + audio index + RRF fusion. So why does Google make a $/minute API for sending the *whole video* to Gemini?

Because for a different class of question, that approach is genuinely better. This notebook lays out the tradeoff with numbers.

## The two architectures (S3 §8.1, §8.2)

**Architecture A — Frame sampling + DIY pipeline (nb09):**
```
video → ffmpeg → 1fps frames + audio → CLIP/Whisper indices → query → top-K timestamps
```
Pros: returns timestamps directly. Searchable. Multi-tenant cost cheap. Multilingual via Whisper.
Cons: loses temporal continuity *between* frames. Can't answer "how does the speaker's tone change" or "what's the relationship between scene 1 and scene 8".

**Architecture B — Native long-context video (Gemini 1.5/2.x):**
```
video → Gemini upload → 1M-token context → ask anything
```
Pros: full temporal awareness. Holistic reasoning over the entire video. Native joint audio+video.
Cons: per-minute pricing. Latency scales with length. No native timestamp output (model has to be asked to produce them, sometimes hallucinated).

We'll send **the same video and the same queries** to both, and tabulate.

## Setup

Set `GOOGLE_API_KEY` in `.env`. Get one free at [aistudio.google.com](https://aistudio.google.com/) — Gemini 2.0 Flash has a generous free tier.

In [ ]:
import os, time
from pathlib import Path
from dotenv import load_dotenv
from google import genai

load_dotenv(dotenv_path="../.env")
assert os.getenv("GOOGLE_API_KEY"), "set GOOGLE_API_KEY in .env"

VIDEO_DIR = Path("../data/video_samples")
videos = sorted(VIDEO_DIR.glob("lecture.*"))
assert videos, "run nb09 first to make sure data/video_samples/lecture.* exists"
VIDEO_PATH = videos[0]

gemini = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

## 1. Upload the video to Gemini

Files API uploads the video, then Gemini processes it (extracts frames at ~1 fps internally, plus audio). The processing step is the upfront latency cost — but it's amortized: subsequent queries reuse the cached file.

In [ ]:
t0 = time.time()
uploaded = gemini.files.upload(file=str(VIDEO_PATH))
print(f"upload + handshake: {time.time() - t0:.1f}s   name: {uploaded.name}")

# Wait for ACTIVE state
t1 = time.time()
while uploaded.state.name == "PROCESSING":
    time.sleep(2)
    uploaded = gemini.files.get(name=uploaded.name)
print(f"processing: {time.time() - t1:.1f}s   state: {uploaded.state.name}")

assert uploaded.state.name == "ACTIVE", f"upload failed: {uploaded.state.name}"

## 2. Three categories of question — same video, different shapes

We're going to ask three deliberately different question types. The hypothesis: each architecture has a sweet spot.

- **Q1 (specific moment retrieval):** *"At what timestamp does the speaker introduce the main topic?"* — DIY pipeline should win. It literally returns timestamps.
- **Q2 (holistic summary):** *"In one paragraph, summarize the entire video."* — Gemini should win. DIY would have to ask GPT-4o over each segment and stitch.
- **Q3 (cross-segment reasoning):** *"How does the speaker's argument develop from beginning to end?"* — Gemini should win clearly. DIY pipeline doesn't preserve the temporal arc.

In [ ]:
QUERIES = [
    ("specific", "At what timestamp (mm:ss) does the speaker introduce the main topic? Quote the line."),
    ("summary",  "In one paragraph, summarize what this video is about."),
    ("arc",      "Describe how the speaker's argument develops from beginning to end. Cite ~3 timestamps that mark turning points."),
]

def ask_gemini(query: str, model: str = "gemini-2.0-flash") -> tuple[str, dict]:
    t0 = time.time()
    resp = gemini.models.generate_content(model=model, contents=[uploaded, query])
    dt = time.time() - t0
    usage = getattr(resp, "usage_metadata", None)
    return resp.text, {
        "latency_s": round(dt, 2),
        "prompt_tokens":     getattr(usage, "prompt_token_count", None),
        "completion_tokens": getattr(usage, "candidates_token_count", None),
    }

gemini_results = {}
for tag, q in QUERIES:
    text, m = ask_gemini(q)
    gemini_results[tag] = {"query": q, "answer": text, "metrics": m}
    print(f"\n=== Gemini · {tag} ({m['latency_s']}s, in={m['prompt_tokens']} out={m['completion_tokens']}) ===")
    print(text)

## 3. The DIY pipeline answers (using nb09's indices)

We re-index here so this notebook stands alone. If you've just run nb09 in the same kernel session, you can skip this and use the existing `frame_vecs` / `audio_vecs`.

In [ ]:
# Reuse nb09's indexing logic — minimal copy
import subprocess, shutil
import torch, torch.nn.functional as F
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
from faster_whisper import WhisperModel
from sentence_transformers import SentenceTransformer
import numpy as np

FRAMES_DIR = VIDEO_DIR / "frames"
AUDIO_PATH = VIDEO_DIR / "audio.wav"
if not list(FRAMES_DIR.glob("*.jpg")):
    FRAMES_DIR.mkdir(exist_ok=True)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(VIDEO_PATH),
                    "-vf", "fps=1", "-q:v", "4", str(FRAMES_DIR / "frame_%05d.jpg")], check=True)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(VIDEO_PATH),
                    "-ac", "1", "-ar", "16000", str(AUDIO_PATH)], check=True)

frame_paths = sorted(FRAMES_DIR.glob("frame_*.jpg"))
frame_seconds = [int(p.stem.split("_")[1]) for p in frame_paths]

device = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

@torch.no_grad()
def clip_image_vecs(paths, b=16):
    out = []
    for i in range(0, len(paths), b):
        imgs = [Image.open(p).convert("RGB") for p in paths[i:i+b]]
        inp = clip_proc(images=imgs, return_tensors="pt").to(device)
        out.append(F.normalize(clip_model.get_image_features(**inp), dim=-1).cpu())
    return torch.cat(out, dim=0)

@torch.no_grad()
def clip_text_vec(t):
    inp = clip_proc(text=[t], return_tensors="pt", padding=True).to(device)
    return F.normalize(clip_model.get_text_features(**inp), dim=-1).cpu()[0]

frame_vecs = clip_image_vecs(frame_paths)

asr = WhisperModel("large-v3", device="cpu", compute_type="int8")
audio_segments = list(asr.transcribe(str(AUDIO_PATH), vad_filter=True)[0])
audio_starts = [s.start for s in audio_segments]
audio_texts = [s.text.strip() for s in audio_segments]

st = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
audio_vecs = st.encode(audio_texts, normalize_embeddings=True, show_progress_bar=False)
print("DIY indices ready")

In [ ]:
from collections import defaultdict
from openai import OpenAI
openai_client = OpenAI()

def search_audio(q, k=5):
    qv = st.encode([q], normalize_embeddings=True)[0]
    idx = np.argsort(-(audio_vecs @ qv))[:k]
    return [(audio_starts[i], audio_texts[i]) for i in idx]

def search_frames(q, k=5):
    sims = (frame_vecs @ clip_text_vec(q)).numpy()
    return [(frame_seconds[i], float(sims[i])) for i in np.argsort(-sims)[:k]]

def diy_answer_specific(query):
    """Specific moment: just return the top audio hit's timestamp + transcript."""
    t0 = time.time()
    hits = search_audio(query, k=1)
    ts, line = hits[0]
    return f"{int(ts // 60):02d}:{int(ts % 60):02d}  — {line}", time.time() - t0

def diy_answer_holistic(query):
    """Holistic: stitch the full transcript and ask GPT-4o-mini to summarize."""
    t0 = time.time()
    full_transcript = "\n".join(
        f"[{int(s // 60):02d}:{int(s % 60):02d}] {t}"
        for s, t in zip(audio_starts, audio_texts)
    )
    r = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Answer based ONLY on the provided timestamped transcript. Cite [mm:ss] where useful."},
            {"role": "user",   "content": f"TRANSCRIPT:\n{full_transcript}\n\nQUESTION: {query}"},
        ],
    )
    return r.choices[0].message.content, time.time() - t0

diy_results = {}
diy_results["specific"] = {"answer": diy_answer_specific(QUERIES[0][1])[0], "latency": diy_answer_specific(QUERIES[0][1])[1]}
for tag, query in QUERIES[1:]:
    ans, dt = diy_answer_holistic(query)
    diy_results[tag] = {"answer": ans, "latency": dt}

for tag, r in diy_results.items():
    print(f"\n=== DIY · {tag} ({r['latency']:.1f}s) ===\n{r['answer']}")

## 4. Side-by-side

In [ ]:
for tag, query in QUERIES:
    print("=" * 80)
    print(f"Q ({tag}): {query}")
    print("-" * 80)
    g = gemini_results[tag]
    print(f"GEMINI ({g['metrics']['latency_s']}s):\n{g['answer']}\n")
    d = diy_results[tag]
    print(f"DIY ({d['latency']:.1f}s):\n{d['answer']}\n")

## 5. Cost comparison

Rough numbers as of early 2026. Adjust as pricing drifts.

**Gemini 2.0 Flash:**
- ~263 video tokens per second of video. A 5-min video → ~80K input tokens per query.
- Input pricing: ~$0.10 per 1M tokens.
- Per-query cost: 80K × $0.10 / 1M ≈ **$0.008** per query (after upload, file is cached).

**DIY pipeline:**
- Upfront indexing (one-time per video): Whisper STT (free, local CPU); CLIP embedding (free, local CPU/GPU).
- Per-query specific-moment: ~free (numpy search).
- Per-query holistic: GPT-4o-mini over the transcript. 5-min lecture transcript ≈ 5K tokens → $0.0008 input + $0.0006 output ≈ **$0.0014** per query.

**At the per-query level, DIY is ~5× cheaper.** Gemini's break-even comes from low-volume use cases where the upfront DIY indexing setup isn't worth it, or from the *quality* gap on holistic / cross-segment reasoning where DIY's transcript-summary approach is genuinely worse.

## 6. Decision rule

What we'd actually deploy:

| Use case | Architecture | Why |
|---|---|---|
| Search-by-text over thousands of lecture videos | **DIY (nb09)** | Per-query cost ~free, returns native timestamps, scales horizontally |
| One-off video summary, podcast → blog post | **Gemini** | No infra to maintain, holistic reasoning is the model's job |
| Cross-segment reasoning ("how did this person's view evolve over the panel") | **Gemini** | DIY's chunked retrieval can't reconstruct the arc |
| Multi-language, code-switched audio | Either; **DIY shows transcripts** which beats Gemini's audio-only summaries for verification | |
| Sensitive content, on-prem only | **DIY** | Whisper + CLIP run anywhere |
| Low-latency "jump to moment" UX | **DIY** | timestamps come back in milliseconds; Gemini takes ~5–15 s per query |

**The hybrid pattern (S3 §8.4):** index with DIY for retrieval, then pass top-K segments to Gemini (or Pegasus, or GPT-4o) for the *generation* step. Best of both.

## End of the lab

We've now covered the full multimodal stack from S3:

- **Images:** how VLMs see them (nb01), CLIP for search (nb02), ColPali for documents (nb03), grounded generation (nb04), multilingual extension (nb05).
- **Audio:** Whisper basics (nb06), code-switching (nb07), full voice agent (nb08).
- **Video:** dual-index DIY (nb09), native long-context comparison (nb10).

Each notebook standalone, each maps cleanly to S3 sections, each is a recordable ~15–25 min explanation. The full series is a portfolio piece students can show off.